In [ ]:
## Evaluating LLMs performance

import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)

# ===================== Konfigurasi =====================
EMO_LABELS = ['anger','antcp','disg','fear','joy','sad','sur','trust']
TAGS = ["GPT", "Ge", "DS", "Ms", "Gr", "Co"]

INPUT_XLSX = "dataset/4. newlabel/LLMs/6indo_LLM.xlsx"
OUTPUT_XLSX = "dataset/4. newlabel/LLMs/indo_metric.xlsx"
PROBA_PREFIX = "proba"   # jika ada probabilitas: anger_proba_GPT, dst.

# ===================== Helpers =====================
def extract_emo_arrays(df: pd.DataFrame, tag: str, labels=EMO_LABELS, proba_prefix=PROBA_PREFIX):
    """
    Ambil y_true, y_pred (biner), y_score (probabilitas jika ada) untuk multilabel emotions.
    - y_true: kolom {lab}_gt
    - y_pred: kolom {lab}_{tag}
    - y_score: kolom {lab}_{proba_prefix}_{tag} (opsional). Jika tak ada, pakai y_pred (biner).
    """
    y_true = df[[f"{lab}_gt" for lab in labels]].to_numpy(dtype=int)
    y_pred = df[[f"{lab}_{tag}" for lab in labels]].to_numpy(dtype=int)

    proba_cols = [f"{lab}_{proba_prefix}_{tag}" for lab in labels]
    if all(col in df.columns for col in proba_cols):
        y_score = df[proba_cols].to_numpy(dtype=float)
    else:
        y_score = y_pred.astype(float)

    return y_true, y_pred, y_score

def evaluate_multilabel(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray, labels=EMO_LABELS):
    """
    Kembalikan dict berisi:
    - subset_accuracy, sample_accuracy
    - micro_precision/recall/f1
    - macro_precision/recall/f1
    - macro_auc
    - auc_<label> untuk tiap label (jika definable, else NaN)
    """
    subset_acc = accuracy_score(y_true, y_pred)
    sample_acc = np.mean((y_true == y_pred).sum(axis=1) / y_true.shape[1])

    micro_prec = precision_score(y_true, y_pred, average='micro', zero_division=0)
    micro_rec  = recall_score(y_true, y_pred, average='micro', zero_division=0)
    micro_f1   = f1_score(y_true, y_pred, average='micro', zero_division=0)

    macro_prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    macro_rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    macro_f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)

    # AUC per label
    per_label_auc = []
    for j in range(y_true.shape[1]):
        col_true = y_true[:, j]
        if len(np.unique(col_true)) >= 2:
            try:
                auc = roc_auc_score(col_true, y_score[:, j])
            except ValueError:
                auc = np.nan
        else:
            auc = np.nan
        per_label_auc.append(auc)

    valid = [a for a in per_label_auc if not np.isnan(a)]
    macro_auc = float(np.mean(valid)) if valid else np.nan

    out = {
        "subset_accuracy": subset_acc,
        "sample_accuracy": sample_acc,
        "micro_precision": micro_prec,
        "micro_recall": micro_rec,
        "micro_f1": micro_f1,
        "macro_precision": macro_prec,
        "macro_recall": macro_rec,
        "macro_f1": macro_f1,
        "macro_auc": macro_auc,
    }
    # tambahkan AUC per label: auc_anger, auc_antcp, ...
    for lab, auc in zip(labels, per_label_auc):
        out[f"auc_{lab}"] = auc
    return out

def evaluate_multiclass(y_true: np.ndarray, y_pred: np.ndarray, average: str = "macro"):
    """Accuracy, Precision, Recall, F1 untuk tugas multiclass (aspect/sentiment)."""
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average=average, zero_division=0),
        "recall":    recall_score(y_true, y_pred, average=average, zero_division=0),
        "f1":        f1_score(y_true, y_pred, average=average, zero_division=0),
    }

# ===================== Evaluasi EMOTIONS utk banyak TAG =====================
def evaluate_emotions_for_tags(df: pd.DataFrame, tags=TAGS, labels=EMO_LABELS) -> pd.DataFrame:
    rows = []
    for tag in tags:
        needed_cols = [f"{lab}_{tag}" for lab in labels] + [f"{lab}_gt" for lab in labels]
        if not all(c in df.columns for c in needed_cols):
            # lewati tag jika kolomnya tidak lengkap
            continue
        y_true, y_pred, y_score = extract_emo_arrays(df, tag, labels=labels)
        m = evaluate_multilabel(y_true, y_pred, y_score, labels=labels)
        m["tag"] = tag
        rows.append(m)
    if not rows:
        return pd.DataFrame(columns=["tag"])
    out_df = pd.DataFrame(rows).set_index("tag")
    return out_df

# ===================== Evaluasi ASPECT & SENTIMENT utk banyak TAG =====================
def evaluate_multiclass_for_tags(df: pd.DataFrame, base_name: str, tags=TAGS, average="macro") -> pd.DataFrame:
    """
    base_name: 'as' untuk aspect, 'sent' untuk sentiment.
    Membaca kolom: f'{base_name}_<TAG>' vs f'{base_name}_gt'
    """
    y_true_col = f"{base_name}_gt"
    if y_true_col not in df.columns:
        # kembalikan DF kosong jika tidak ada GT
        return pd.DataFrame(columns=["tag","accuracy","precision","recall","f1"]).set_index("tag")

    out = []
    for tag in tags:
        y_pred_col = f"{base_name}_{tag}"
        if y_pred_col not in df.columns:
            continue
        y_true = df[y_true_col].to_numpy(dtype=int)
        y_pred = df[y_pred_col].to_numpy(dtype=int)
        m = evaluate_multiclass(y_true, y_pred, average=average)
        m["tag"] = tag
        out.append(m)

    if not out:
        return pd.DataFrame(columns=["tag","accuracy","precision","recall","f1"]).set_index("tag")
    return pd.DataFrame(out).set_index("tag")

def round_df(df: pd.DataFrame, digits=4):
    """Bulatkan nilai numerik agar rapi saat disimpan."""
    if df is None or df.empty:
        return df
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].round(digits)
    return df

# ===================== Main =====================
if __name__ == "__main__":
    # 1) Baca data
    df = pd.read_excel(INPUT_XLSX)

    # 2) Hitung ringkasan
    emo_summary  = evaluate_emotions_for_tags(df, tags=TAGS, labels=EMO_LABELS)
    as_summary   = evaluate_multiclass_for_tags(df, base_name="as",   tags=TAGS, average="macro")
    sent_summary = evaluate_multiclass_for_tags(df, base_name="sent", tags=TAGS, average="macro")

    # 3) Bulatkan angka
    emo_summary  = round_df(emo_summary, digits=4)
    as_summary   = round_df(as_summary, digits=4)
    sent_summary = round_df(sent_summary, digits=4)

    # 4) Simpan ke satu file Excel, 3 sheet
    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        emo_summary.to_excel(writer, sheet_name="emotion")
        as_summary.to_excel(writer,  sheet_name="aspect")
        sent_summary.to_excel(writer, sheet_name="sentiment")

    print(f"Saved metrics to: {OUTPUT_XLSX}")
